# 04. LTV по сегментам и unit-экономика

В прошлых ноутбуках мы выяснили, что клиенты с высоким первым чеком возвращаются чаще. Теперь надо понять, насколько эта разница реально окупается в деньгах. Иначе говоря, разобраться с unit-экономикой каждого сегмента.

Что считаем:
1. LTV (lifetime value) каждого клиента: суммарная выручка с него за весь период наблюдения.
2. Среднее число заказов и средний чек по сегментам.
3. Payback по упрощённой модели: при каком CAC и марже сегмент окупается.

Главный вывод одной строкой: сегмент 'высокий первый чек' приносит за период наблюдения примерно в 4 раза больше денег, чем 'низкий', и это уже не про retention, а про реальную выручку.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_parquet('../data/clean.parquet')
customers = pd.read_parquet('../data/customers_labeled.parquet')
print(f'Транзакций: {len(df):,}')
print(f'Клиентов в сегментации: {len(customers):,}')

## Считаем LTV для каждого клиента

В данном проекте под LTV я понимаю простое: суммарную выручку с клиента за весь доступный период (два года). Это не настоящий 'lifetime', а наблюдаемая часть, и для бизнес-разговора этого достаточно. Полноценный LTV считается через сложные модели (BG/NBD и Gamma-Gamma), но для интерн-проекта простого подхода хватает, и его проще защитить.

In [ ]:
ltv = (
    df.groupby('Customer ID')
    .agg(
        ltv_revenue=('Revenue', 'sum'),
        n_orders=('Invoice', 'nunique'),
        first_purchase=('InvoiceDate', 'min'),
        last_purchase=('InvoiceDate', 'max'),
    )
    .reset_index()
)
ltv['avg_order_value'] = ltv['ltv_revenue'] / ltv['n_orders']
ltv['lifetime_days'] = (ltv['last_purchase'] - ltv['first_purchase']).dt.days

ltv.head()

In [ ]:
# Объединяем с сегментами по первому чеку
ltv = ltv.merge(
    customers[['Customer ID', 'check_segment', 'returned']],
    on='Customer ID',
    how='left',
)
ltv.head()

## Сравниваем LTV по сегментам

Чтобы цифры не страдали от выбросов, считаем сразу и среднее, и медиану. Если они близки, это значит распределение более-менее симметричное, если далеко друг от друга, в выборке есть тяжёлый хвост.

In [ ]:
segment_stats = (
    ltv.groupby('check_segment')
    .agg(
        n_customers=('Customer ID', 'count'),
        ltv_mean=('ltv_revenue', 'mean'),
        ltv_median=('ltv_revenue', 'median'),
        avg_orders=('n_orders', 'mean'),
        avg_aov=('avg_order_value', 'mean'),
        avg_lifetime_days=('lifetime_days', 'mean'),
    )
    .round(1)
)
segment_stats

Что из таблицы видно сразу:
1. Средний LTV у сегмента с высоким первым чеком кратно больше. Не на проценты, а в разы.
2. Среднее число заказов у них тоже выше, то есть они не просто разово купили на много, а ещё и возвращались чаще.
3. Средняя длина 'жизни' клиента (от первой до последней покупки) у высокого сегмента в разы больше.

По сути, мы получили независимое подтверждение результатов t-теста, но уже в деньгах.

In [ ]:
# Распределение LTV по сегментам, лог-шкала чтобы не убил хвост
fig, ax = plt.subplots(figsize=(10, 5))
for seg, color in zip(
    ltv['check_segment'].dropna().unique(),
    ['#C44E52', '#55A868'],
):
    subset = ltv.loc[ltv['check_segment'] == seg, 'ltv_revenue']
    ax.hist(np.log10(subset[subset > 0]), bins=40, alpha=0.55, label=seg, color=color)

ax.set_title('Распределение LTV по сегментам (лог-шкала)')
ax.set_xlabel('log10(LTV в £)')
ax.set_ylabel('Кол-во клиентов')
ax.legend()
plt.tight_layout()
plt.savefig('../images/ltv_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

Лог-шкала по горизонтали нужна потому, что у LTV огромный разброс: от пары фунтов до сотен тысяч. На обычной шкале гистограмма превращается в одну палку у нуля. Лог-шкала растягивает картинку и даёт сравнить форму распределений.

## Кумулятивный вклад в выручку: правило Парето

Простая, но всегда зрелищная штука: отсортируем клиентов по LTV и посмотрим, какая доля клиентов даёт какую долю выручки.

In [ ]:
ltv_sorted = ltv.sort_values('ltv_revenue', ascending=False).reset_index(drop=True)
ltv_sorted['cum_revenue_pct'] = ltv_sorted['ltv_revenue'].cumsum() / ltv_sorted['ltv_revenue'].sum() * 100
ltv_sorted['cum_customers_pct'] = (ltv_sorted.index + 1) / len(ltv_sorted) * 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ltv_sorted['cum_customers_pct'], ltv_sorted['cum_revenue_pct'], color='#4C72B0', linewidth=2)
ax.plot([0, 100], [0, 100], '--', color='gray', linewidth=1, label='линия равенства')
ax.axvline(x=20, color='red', linestyle=':', alpha=0.6)
ax.axhline(y=ltv_sorted.loc[int(len(ltv_sorted) * 0.20), 'cum_revenue_pct'], color='red', linestyle=':', alpha=0.6)

share_at_20 = ltv_sorted.loc[int(len(ltv_sorted) * 0.20), 'cum_revenue_pct']
ax.set_title(f'Концентрация выручки: топ-20% клиентов дают {share_at_20:.0f}% выручки')
ax.set_xlabel('Доля клиентов, %')
ax.set_ylabel('Доля выручки, %')
ax.legend()
plt.tight_layout()
plt.savefig('../images/pareto_revenue.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Топ-20% клиентов приносят {share_at_20:.1f}% всей выручки')

Стандартная картинка для розницы: меньшая часть клиентов даёт большую часть денег. Это не баг, а норма, и любое решение про маркетинговый бюджет должно её учитывать. Терять в этом топе одного клиента дороже, чем потерять десяток в среднем сегменте.

## Простая модель unit-экономики

Дальше будет упражнение с допущениями. У нас в датасете нет реальных данных по CAC и марже, поэтому возьмём правдоподобные числа и посмотрим, как окупаемость зависит от сегмента.

Допущения:
- CAC (стоимость привлечения одного клиента) = £20. Цифра условная, но порядка для британской розницы похожая.
- Валовая маржа = 30% от выручки. Тоже типично для ритейла, без премиум-сегмента.
- Под 'окупаемостью' понимаю отношение валовой маржи к CAC. Если оно больше 1, сегмент уже отбил привлечение.

In [ ]:
CAC = 20.0  # £
GROSS_MARGIN = 0.30

unit_econ = (
    ltv.groupby('check_segment')
    .agg(ltv_mean=('ltv_revenue', 'mean'), ltv_median=('ltv_revenue', 'median'))
    .reset_index()
)
unit_econ['gross_profit_mean'] = unit_econ['ltv_mean'] * GROSS_MARGIN
unit_econ['gross_profit_median'] = unit_econ['ltv_median'] * GROSS_MARGIN
unit_econ['payback_x_mean'] = unit_econ['gross_profit_mean'] / CAC
unit_econ['payback_x_median'] = unit_econ['gross_profit_median'] / CAC
unit_econ.round(2)

Как читать таблицу:
- payback_x_mean это во сколько раз валовая маржа со среднего клиента сегмента превышает CAC. Например, 5.0 означает, что в среднем мы вернули CAC пятикратно.
- payback_x_median полезнее как 'честная' цифра: средний задирается оптовиками, а медиана говорит про типичного клиента.

Главное: даже по медиане сегмент с высоким первым чеком окупается с большим запасом, а низкий находится близко к границе. Это значит, что любой рост CAC (например, аукционная инфляция в рекламе) сначала ударит именно по низкому сегменту, и его экономика может стать отрицательной первой.

In [ ]:
# Чувствительность: при каком CAC сегмент перестанет окупаться (по медиане)
cac_grid = np.linspace(5, 100, 50)
fig, ax = plt.subplots(figsize=(10, 5))

for seg, color in zip(unit_econ['check_segment'], ['#C44E52', '#55A868']):
    median_ltv = unit_econ.loc[unit_econ['check_segment'] == seg, 'ltv_median'].iloc[0]
    payback = (median_ltv * GROSS_MARGIN) / cac_grid
    ax.plot(cac_grid, payback, label=seg, color=color, linewidth=2)

ax.axhline(y=1, color='black', linestyle='--', alpha=0.6, label='граница окупаемости (payback = 1)')
ax.set_xlabel('CAC, £')
ax.set_ylabel('Payback (валовая маржа / CAC), по медиане')
ax.set_title('Чувствительность окупаемости к стоимости привлечения')
ax.legend()
plt.tight_layout()
plt.savefig('../images/cac_sensitivity.png', dpi=120, bbox_inches='tight')
plt.show()

Из этого графика выводится одна полезная для разговора цифра: тот CAC, при котором низкий сегмент перестаёт окупаться. Если бизнес видит, что аукционная цена клика растёт и реальный CAC приближается к этой границе, надо либо переставать лить трафик на условия, дающие низкочековых клиентов, либо повышать средний чек первой покупки (через бандлы и порог бесплатной доставки).

## Сохраняем расчёт LTV для следующих ноутбуков

In [ ]:
ltv.to_parquet('../data/ltv.parquet', index=False)
print('Сохранено: data/ltv.parquet')

## Что мы поняли в этом ноутбуке

1. Сегмент с высоким первым чеком приносит кратно больше денег за весь период наблюдения, причём не только из-за более высокого AOV, но и из-за большего числа повторных покупок и большей продолжительности 'жизни'.
2. Концентрация выручки очень неравномерная (правило Парето в чистом виде), и любые решения по бюджету должны это учитывать.
3. По упрощённой unit-экономике высокий сегмент окупается с большим запасом, низкий находится ближе к границе. Это меняет акцент рекомендаций из ноутбука 03: триггерные кампании в низком сегменте имеют смысл, но цена ошибки в этом сегменте выше, и тестировать их надо аккуратно.